In [ ]:
"""TV3 - Làm sạch & kiểm tra dataset [P] (Electric Vehicle Population Data).
Chạy:  python TV3_lam_sach_P.py
- Phần A chạy trên file sạch (11 cột) -> có sẵn.
- Phần B chỉ chạy nếu có file gốc 17 cột (đặt tên RAW_PATH bên dưới) -> cho số dòng trước/sau từng bước."""
import os, pandas as pd, numpy as np

CLEAN_PATH = "EV_Population_Washington_Clean__1_.csv"
RAW_PATH   = "Electric_Vehicle_Population_Data.csv"      # file gốc 17 cột (nếu còn)
DROP_COLS  = ["VIN (1-10)", "Legislative District", "DOL Vehicle ID",
              "Vehicle Location", "Electric Utility", "2020 Census Tract"]

# ================= PHẦN B (tùy chọn): file gốc =================
if os.path.exists(RAW_PATH):
    raw = pd.read_csv(RAW_PATH, dtype={"Postal Code": "string"})
    print("B1. File gốc:", raw.shape)
    wa = raw[raw["State"].str.strip().str.upper() == "WA"]
    print("B2. Sau lọc State = WA:", len(wa), "dòng (loại", len(raw) - len(wa), ")")
    print("B3. Sau bỏ 6 cột:", wa.drop(columns=DROP_COLS, errors="ignore").shape)
    if "DOL Vehicle ID" in raw.columns:
        dup_id = wa["DOL Vehicle ID"].duplicated().sum()
        print("B4. DOL Vehicle ID lặp:", dup_id,
              "-> ghi 'đã kiểm tra, không có bản ghi lặp'" if dup_id == 0 else "-> cần xử lý/ghi rõ")
else:
    print("(Không có file gốc: bỏ qua phần B; chỉ báo cáo kiểm tra trên file sạch.)")

# ================= PHẦN A: file sạch =================
P = pd.read_csv(CLEAN_PATH)
print("\nA0. Kích thước:", P.shape, "| State:", P["State"].unique())

# A1. Khoảng trắng thừa & khác biệt hoa/thường
text_cols = ["County", "City", "State", "Make", "Model", "Electric Vehicle Type",
             "Clean Alternative Fuel Vehicle (CAFV) Eligibility"]
for c in text_cols:
    s = P[c].dropna()
    print(f"A1. {c[:28]:28s} khoang trang thua: {(s != s.str.strip()).sum():d} | "
          f"so nhan: {s.nunique()} -> sau UPPER/strip: {s.str.strip().str.upper().nunique()}")
city = P.assign(K=P["City"].str.strip().str.upper())
dupl = city.groupby("K")["City"].nunique(); print("A1. City khac hoa/thuong:", dupl[dupl > 1].index.tolist())
print(P[P["City"].str.upper() == "MCCLEARY"].groupby(["City", "County"]).size())

# A2. Chuẩn hóa nhãn
P["Type"] = np.where(P["Electric Vehicle Type"].str.contains("BEV"), "BEV", "PHEV")
P["Make"] = P["Make"].str.strip().str.upper()
P["Model"] = P["Model"].str.strip().str.upper()
P["County"] = P["County"].str.strip()
P["City"] = P["City"].str.strip().str.upper().str.title().replace({"Mccleary": "McCleary"})
print("A2. So City sau gop:", P["City"].nunique(), "| So County:", P["County"].nunique())

# A3. Ép kiểu (file CSV lưu 'Postal Code', 'Model Year' dạng 99344.0, 2020.0)
P["Postal Code"] = P["Postal Code"].astype("Int64").astype("string").str.zfill(5)
P["Model Year"] = P["Model Year"].astype("Int64")
P["Electric Range"] = P["Electric Range"].astype("Int64")
P["Base MSRP"] = P["Base MSRP"].astype("Int64")
print(P.dtypes)

# A4. Giá trị hợp lệ
print("\nA4. Model Year:", P["Model Year"].min(), "-", P["Model Year"].max(), "| truoc 2010:", (P["Model Year"] < 2010).sum())
print(P[P["Model Year"] < 2010].groupby(["Make", "Model", "Model Year"]).size())
print("A4. Electric Range:", P["Electric Range"].min(), "-", P["Electric Range"].max(), "| thieu:", P["Electric Range"].isna().sum())
print("A4. Postal Code:", P["Postal Code"].min(), "-", P["Postal Code"].max(), "| do dai:", P["Postal Code"].str.len().unique())
print("A4. Base MSRP:", P["Base MSRP"].min(), "-", P["Base MSRP"].max(), "| co gia tri:", P["Base MSRP"].notna().sum())

# A5. Dòng trùng: CHỈ KIỂM TRA, KHÔNG drop_duplicates
cols = [c for c in P.columns if c != "Type"]
extra = P.duplicated(subset=cols).sum()
print(f"\nA5. Dong giong het tren 11 cot: {extra:,} | neu drop se con {len(P)-extra:,} dong ({extra/len(P):.1%} bi mat)")
grp = P.groupby(cols, dropna=False).size().sort_values(ascending=False)
print("A5. So nhom >1 dong:", (grp > 1).sum(), "| nhom lon nhat:", grp.iloc[0]); print(grp.head(3))
print("=> Giu nguyen", len(P), "dong (gia dinh moi dong la mot xe).")